In [0]:
# 02 - Data Understanding (CRISP-DM)
Explore `claims_clean`: target balance, missingness, distributions, fraud-rate by provider/attorney/state, geography, occupancy, and the adjuster notes (text).

Use the figures here in your **Business Understanding Document -> Data Understanding** section. Fill in the *Findings* cells after you run each section.

In [0]:
SRC_TABLE = "workspace.insurance_fraud.auto_insurance_fraud"   # READ-ONLY warehouse table
DM = "workspace.dm_final_project"                               # ALL derived work goes here
SEED = 42
assert "insurance_fraud" not in DM, "Never write to the warehouse schema"

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

df = spark.table(f"{DM}.claims_clean").toPandas()
df["claim_date"] = pd.to_datetime(df["claim_date"])
ORG, OPP = "is_organized_fraud", "is_opportunistic_fraud"
df["fraud_type"] = np.select([(df[ORG] == 1) & (df[OPP] == 1), df[ORG] == 1, df[OPP] == 1], ["both", "organized", "opportunistic"], "legitimate")
print(df.shape)
df.head()

In [0]:
## 1. Entity Relationship Diagram
The warehouse exposes one denormalized `auto_insurance_fraud` table; the *logical* ERD below shows the entities implied by `medical_provider`, `attorney_firm` and the incident location. Copy the Mermaid block into docs that render it, or use the matplotlib rendering.

```mermaid
erDiagram
    MEDICAL_PROVIDER ||--o{ CLAIMS : treats
    ATTORNEY_FIRM    ||--o{ CLAIMS : represents
    STATE            ||--o{ CLAIMS : "incident in"
    CLAIMS {
        string claim_id PK
        date claim_date
        string state FK
        int insured_age
        int policy_inception_days
        int num_people_in_car
        int num_different_surnames
        int report_delay_hours
        string medical_provider FK
        string attorney_firm FK
        float incident_latitude
        float incident_longitude
        int incident_time_hour
        int prior_claims_6mo
        int vehicle_age_years
        float damage_amount
        int claim_amendments
        float initial_damage_amount
        int medical_visits_count
        int days_to_attorney
        int settlement_inquiries
        int is_organized_fraud
        int is_opportunistic_fraud
        text claim_notes
    }
```

In [0]:
from matplotlib.patches import FancyBboxPatch
def draw_entity(ax, x, y, title, fields, w=3.4, lh=0.3):
    h = lh * (len(fields) + 1)
    ax.add_patch(FancyBboxPatch((x, y - h), w, h, boxstyle="round,pad=0.02", fc="#f4f7fb", ec="#1f3b63"))
    ax.text(x + w / 2, y - lh / 2, title, ha="center", va="center", weight="bold")
    for i, f in enumerate(fields):
        ax.text(x + 0.1, y - lh * (i + 1.5), f, va="center", fontsize=7.5, family="monospace")
    return dict(left=(x, y - h / 2), right=(x + w, y - h / 2), cy=y - h / 2)

claims_fields = ["PK claim_id","claim_date","FK state","insured_age","policy_inception_days","num_people_in_car",
    "num_different_surnames","report_delay_hours","FK medical_provider","FK attorney_firm","incident_latitude",
    "incident_longitude","incident_time_hour","prior_claims_6mo","vehicle_age_years","damage_amount",
    "claim_amendments","initial_damage_amount","medical_visits_count","days_to_attorney","settlement_inquiries",
    "is_organized_fraud (target)","is_opportunistic_fraud (target)","claim_notes (text)"]
fig, ax = plt.subplots(figsize=(11, 8))
c = draw_entity(ax, 0, 8.4, "CLAIMS", claims_fields)
p = draw_entity(ax, 7, 8.4, "MEDICAL_PROVIDER", ["PK medical_provider (name)", "(derived) claim_volume", "(derived) fraud_rate"])
a = draw_entity(ax, 7, 5.6, "ATTORNEY_FIRM", ["PK attorney_firm (name)", "(derived) claim_volume", "(derived) fraud_rate"])
s = draw_entity(ax, 7, 2.8, "STATE", ["PK state", "(derived) fraud_rate"])
for tgt, lab in [(p, "1 : N"), (a, "1 : N"), (s, "1 : N")]:
    ax.annotate("", xy=c["right"], xytext=tgt["left"], arrowprops=dict(arrowstyle="-", color="#1f3b63"))
    ax.text((c["right"][0] + tgt["left"][0]) / 2, (c["right"][1] + tgt["left"][1]) / 2 + 0.1, lab, fontsize=8)
ax.set_xlim(-0.3, 10.7); ax.set_ylim(0, 8.7); ax.axis("off"); ax.set_title("Logical ERD - IronKey claims")
plt.tight_layout(); plt.show()

## 2. Shape, types, missingness

In [0]:
summary = pd.DataFrame({"dtype": df.dtypes.astype(str), "nulls": df.isna().sum(), "null_pct": (df.isna().mean() * 100).round(2), "n_unique": df.nunique()})
display(summary)
display(df.describe().T)

In [0]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
fig, ax = plt.subplots(figsize=(8, 0.4 * max(len(miss), 3) + 1))
miss.mul(100).plot.barh(ax=ax); ax.set_xlabel("% missing"); ax.set_title("Missing values by column"); plt.show()

# Is missingness informative? (fraud rate when the field is missing vs present)
rows = []
for col in miss.index:
    for t in [ORG, OPP]:
        rows.append(dict(column=col, target=t, rate_when_missing=df.loc[df[col].isna(), t].mean(), rate_when_present=df.loc[df[col].notna(), t].mean()))
display(pd.DataFrame(rows))

> **Findings (fill in):** which fields are missing by design (e.g. `initial_damage_amount` only when amended, `days_to_attorney`/`attorney_firm` when no attorney) vs. data-quality gaps. Informative missingness => we will keep missing-indicators in Data Preparation.

## 3. Target balance & overlap

In [0]:
bal = df["fraud_type"].value_counts().rename_axis("fraud_type").reset_index(name="claims")
bal["pct"] = (100 * bal["claims"] / len(df)).round(2)
display(bal)
fig, ax = plt.subplots(figsize=(6, 3.5)); sns.barplot(data=bal, x="fraud_type", y="claims", ax=ax); ax.set_title("Claim class balance"); plt.show()

N_ANNUAL = 150_000
for t in [ORG, OPP]:
    print(f"{t}: {df[t].mean():.2%} of claims -> ~{N_ANNUAL * df[t].mean():,.0f} claims/yr at IronKey volume")
print("Avg damage legit vs fraud:"); display(df.groupby("fraud_type")["damage_amount"].agg(["count", "mean", "median", "sum"]).round(0))

> **Findings (fill in):** severe class imbalance => use PR-AUC / recall at fixed review capacity, class weights, never accuracy.

## 4. Numeric distributions by fraud type

In [0]:
num_cols = ["insured_age","policy_inception_days","num_people_in_car","num_different_surnames","report_delay_hours",
            "incident_time_hour","prior_claims_6mo","vehicle_age_years","damage_amount","claim_amendments",
            "initial_damage_amount","medical_visits_count","days_to_attorney","settlement_inquiries"]
num_cols = [c for c in num_cols if c in df.columns]
ncols = 3; nrows = int(np.ceil(len(num_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.6 * nrows))
for ax, c in zip(axes.ravel(), num_cols):
    sns.boxplot(data=df, x="fraud_type", y=c, ax=ax, showfliers=False, order=["legitimate","opportunistic","organized","both"] if "both" in df.fraud_type.values else ["legitimate","opportunistic","organized"])
    ax.set_title(c); ax.set_xlabel("")
for ax in axes.ravel()[len(num_cols):]: ax.axis("off")
plt.tight_layout(); plt.show()
display(df.groupby("fraud_type")[num_cols].mean().round(2).T)

In [0]:
# Correlation of each numeric feature with each target
corr = df[num_cols + [ORG, OPP]].corr()[[ORG, OPP]].drop([ORG, OPP]).sort_values(ORG)
fig, ax = plt.subplots(figsize=(6, 6)); sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax); ax.set_title("Correlation with targets"); plt.show()
fig, ax = plt.subplots(figsize=(10, 8)); sns.heatmap(df[num_cols].corr(), cmap="RdBu_r", center=0, ax=ax); ax.set_title("Feature-feature correlation (multicollinearity check)"); plt.show()

## 5. Derived signals (preview of feature engineering)

In [0]:
d = df.copy()
init = d["initial_damage_amount"].fillna(d["damage_amount"])
d["damage_inflation_ratio"] = d["damage_amount"] / init.replace(0, np.nan)
d["surname_ratio"] = d["num_different_surnames"] / d["num_people_in_car"].replace(0, np.nan)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.boxplot(data=d, x="fraud_type", y="damage_inflation_ratio", showfliers=False, ax=axes[0]); axes[0].set_title("Damage inflation ratio")
sns.boxplot(data=d, x="fraud_type", y="surname_ratio", ax=axes[1]); axes[1].set_title("Different surnames / occupants")
sns.boxplot(data=d, x="fraud_type", y="days_to_attorney", showfliers=False, ax=axes[2]); axes[2].set_title("Days to attorney")
plt.tight_layout(); plt.show()

print("Organized fraud rate by occupants x distinct surnames")
display(pd.crosstab(d["num_people_in_car"], d["num_different_surnames"], values=d[ORG], aggfunc="mean").round(3))

## 6. Provider, attorney and state networks

In [0]:
def rate_table(col, target, min_n=30, top=15):
    g = df.groupby(col)[target].agg(n="count", frauds="sum", rate="mean").query("n >= @min_n").sort_values("rate", ascending=False)
    return g.head(top)

for col in ["medical_provider", "attorney_firm"]:
    for t in [ORG, OPP]:
        g = rate_table(col, t)
        print(f"\nTop {col} by {t} rate (min 30 claims) | overall rate {df[t].mean():.2%}")
        display(g.round(3))
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, col in zip(axes, ["medical_provider", "attorney_firm"]):
    g = rate_table(col, ORG, top=12)
    ax.barh(g.index[::-1], g["rate"][::-1]); ax.set_title(f"Organized-fraud rate: {col}"); ax.set_xlabel("rate")
plt.tight_layout(); plt.show()

st = df.groupby("state")[[ORG, OPP]].mean().mul(100).round(2).sort_values(ORG, ascending=False)
display(st)

In [0]:
# Provider-attorney co-occurrence among organized-fraud claims (ring detection preview)
ring = df[df[ORG] == 1].groupby(["medical_provider", "attorney_firm"]).size().sort_values(ascending=False).head(15)
display(ring.rename("organized_fraud_claims").reset_index())

## 7. Geography and time

In [0]:
geo = df.dropna(subset=["incident_latitude", "incident_longitude"])
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
for ax, t, title in zip(axes, [ORG, OPP], ["Organized fraud", "Opportunistic fraud"]):
    ax.scatter(geo["incident_longitude"], geo["incident_latitude"], s=2, c="lightgray", alpha=.3)
    f = geo[geo[t] == 1]; ax.scatter(f["incident_longitude"], f["incident_latitude"], s=6, c="crimson", alpha=.6)
    ax.set_title(title); ax.set_xlabel("longitude")
axes[0].set_ylabel("latitude"); plt.tight_layout(); plt.show()

m = df.set_index("claim_date").resample("M")[[ORG, OPP]].mean().mul(100)
m.plot(figsize=(10, 3.5), title="Monthly fraud rate (%)"); plt.show()
h = df.groupby("incident_time_hour")[[ORG, OPP]].mean().mul(100)
h.plot(kind="bar", figsize=(10, 3.5), title="Fraud rate (%) by incident hour"); plt.show()

## 8. Text understanding - `claim_notes`
Data-driven n-grams ranked by log-odds ratio (fraud vs. not). These guide the NLP features in the next notebook; do **not** rely only on the examples in the assignment.

In [0]:
from sklearn.feature_extraction.text import CountVectorizer
notes = df["claim_notes"].fillna("")
print("Notes missing/blank:", (notes == "").mean().round(3))
print("Avg words per note by class:"); display(df.assign(w=notes.str.split().str.len()).groupby("fraud_type")["w"].mean().round(1))
for t in ["organized", "opportunistic"]:
    pass
display(df.loc[df[ORG] == 1, "claim_notes"].head(5).to_frame())
display(df.loc[df[OPP] == 1, "claim_notes"].head(5).to_frame())

cv = CountVectorizer(ngram_range=(2, 3), min_df=25, stop_words="english", binary=True)
X = cv.fit_transform(notes)
vocab = np.array(cv.get_feature_names_out())

def log_odds(y, top=20, alpha=1.0):
    y = np.asarray(y).astype(bool)
    pos = np.asarray(X[y].sum(axis=0)).ravel() + alpha; neg = np.asarray(X[~y].sum(axis=0)).ravel() + alpha
    lo = np.log(pos / pos.sum()) - np.log(neg / neg.sum())
    cnt = np.asarray(X.sum(axis=0)).ravel()
    out = pd.DataFrame({"ngram": vocab, "log_odds": lo, "docs": cnt}).sort_values("log_odds", ascending=False)
    return out.head(top)

for t in [ORG, OPP]:
    r = log_odds(df[t]); print(f"\nTop n-grams for {t}"); display(r.round(2))
    fig, ax = plt.subplots(figsize=(7, 5)); ax.barh(r["ngram"][::-1], r["log_odds"][::-1]); ax.set_title(f"Log-odds: {t}"); plt.show()

> **Leakage check:** if top n-grams contain words like "fraud", "SIU", "EUO", "denied", "referred", the notes were written *after* the fraud decision and must be excluded. The preparation notebook filters such terms.

## 9. Summary of findings (fill in for your Data Understanding section)
| Question | Finding |
|---|---|
| Class balance | |
| Fields missing by design vs. quality | |
| Strongest numeric signals | |
| Provider / attorney concentration | |
| Geographic clustering | |
| Text signals beyond the assignment examples | |
| Data-quality risks / leakage risks | |